# GeoLife CP1 — Trajectory Config Deep-Dive Case Studies

**Mục tiêu:** giải thích bốn baseline config ở cấp trajectory và kiểm tra **behavior / trade-off** của từng parameter:

```text
max_gap_s              : 300 s
distance_threshold_m   : 200 m
min_dwell_s            : 20 min
hard_speed_guard_kmh   : 1200 km/h
```

Notebook này là **case-study / explainability layer**, không phải nơi tự động freeze production config.

Mỗi case sẽ có:

1. sensitivity comparison khi chỉ thay **một parameter**;
2. ghi rõ các parameter còn lại đang được **fix ở baseline nào**;
3. diagnostic đúng với semantics của parameter;
4. một cell tự sinh:
   - **Case này support điều gì?**
   - **Case này chưa thể kết luận điều gì?**

Cuối notebook **không tạo `FINAL_CONFIG`**. Việc freeze CP1 phải dựa vào population-level sensitivity ở notebook **02b**. Nếu CP1 baseline thực sự thay đổi sau 02b, các downstream artifact/cache ở notebook **03** phải được regenerate.

> Đây là engineering diagnostics trên GeoLife, không phải ground-truth optimization.


In [ ]:
# Optional private runtime setup.
# Do NOT hard-code or commit MAPTILER_API_KEY in this notebook.
# Example (run in a temporary/private cell if needed):
# import os
# os.environ["MAPTILER_API_KEY"] = "YOUR_KEY"


## 0. Setup

Notebook cố ý đứng độc lập để PR dễ review:

- dùng cùng GeoLife parser / cleaning / staypoint code của project;
- case selection deterministic, không chọn trajectory bằng mắt;
- cache scan riêng để Run All không phải tính lại nếu dữ liệu không đổi;
- MapTiler key chỉ đọc từ environment, **không hard-code trong notebook**;
- output của mỗi case được diễn giải theo đúng mức evidence mà case đó support.

Các threshold phụ dùng để **diagnose** không được tự động diễn giải thành transport-mode classifier.

### Baseline dùng để isolate từng parameter

```text
same_second_radius_m   = 10 m
max_gap_s              = 300 s
distance_threshold_m   = 200 m
min_dwell_s            = 1200 s = 20 min
hard_speed_guard_kmh   = 1200 km/h
```

Khi một Case thay một parameter, bốn giá trị còn lại giữ nguyên baseline trừ khi cell ghi rõ khác đi.


In [ ]:
from pathlib import Path
from time import perf_counter
from IPython.display import display, Markdown
from matplotlib.patches import Circle
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_URL = "https://github.com/tanh1c/geolife.git"
REPO_BRANCH = os.environ.get("GEOLIFE_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("GEOLIFE_REPO_DIR", "/tmp/geolife"))
CACHE_DIR = Path(
    os.environ.get(
        "GEOLIFE_CACHE_DIR",
        "/mnt/geolife-data/cache/cp1_staypoints",
    )
)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

BASELINE = {
    "same_second_radius_m": 10.0,
    "max_gap_s": 300.0,
    "hard_speed_guard_kmh": 1200.0,
    "distance_threshold_m": 200.0,
    "min_dwell_s": 1200.0,
}

def resolve_data_root() -> Path:
    env_root = os.environ.get("GEOLIFE_DATA_ROOT")
    candidates = ([Path(env_root)] if env_root else []) + [
        Path("/mnt/geolife-data/extracted/Geolife Trajectories 1.3/Data"),
        Path("/mnt/geolife-data/Data"),
        Path("/mnt/geolife-data/extracted/Data"),
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
            return candidate

    volume_root = Path("/mnt/geolife-data")
    if volume_root.is_dir():
        for candidate in sorted(volume_root.glob("**/Data")):
            if candidate.is_dir() and any(candidate.glob("*/Trajectory/*.plt")):
                return candidate

    raise FileNotFoundError("GeoLife Data folder not found")

def ensure_repo() -> None:
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )

DATA_ROOT = resolve_data_root()
ensure_repo()

for candidate in (REPO_DIR, REPO_DIR / "src"):
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from notebooks.eda_core import read_plt
from geolife.geo.distance import haversine_m
from geolife.staypoints import (
    clean_trajectory,
    clean_trajectory_with_audit,
    detect_staypoints,
)

files = sorted(DATA_ROOT.glob("*/Trajectory/*.plt"))

print("Repo branch:", REPO_BRANCH)
print("Data root:", DATA_ROOT)
print("Trajectory files:", f"{len(files):,}")
print("Cache:", CACHE_DIR)
print("Baseline:", BASELINE)

## 1. Deterministic case scan

### Vì sao không chọn case bằng mắt?

Nếu mở nhiều trajectory rồi chọn một file làm baseline trông đẹp, case study dễ trở thành cherry-pick.

Ta dùng rule:

1. phủ tất cả user;
2. tối đa 5 files / user;
3. user có nhiều file thì lấy trải đều theo lịch sử;
4. scan ba one-factor-at-a-time axes:
   - gap: `120 / 300 / 600 s`;
   - radius: `100 / 200 / 300 m`;
   - dwell: `10 / 20 / 30 min`;
5. trong nhóm có detector behavior thay đổi, ưu tiên trajectory ít point hơn để figure dễ đọc.

Population sensitivity chính thức vẫn nằm ở `02b`; scan này chỉ phục vụ **case discovery**.

In [ ]:
MAX_FILES_PER_USER = 5
CASE_SCAN_CACHE = CACHE_DIR / "trajectory_config_case_scan_v3_deep_dive.pkl"

def user_id_from_path(path: Path) -> str:
    return path.parent.parent.name

def build_user_stratified_sample(paths, max_files_per_user=5):
    by_user = {}
    for path in paths:
        by_user.setdefault(user_id_from_path(path), []).append(path)

    sample = []
    for user_id in sorted(by_user):
        user_paths = sorted(by_user[user_id])
        k = min(max_files_per_user, len(user_paths))
        if k == len(user_paths):
            selected = user_paths
        else:
            idx = np.unique(
                np.linspace(0, len(user_paths) - 1, num=k, dtype=int)
            )
            selected = [user_paths[int(i)] for i in idx]
        sample.extend((user_id, path) for path in selected)
    return sample

sample = build_user_stratified_sample(files, MAX_FILES_PER_USER)

print("Users covered:", len({user_id for user_id, _ in sample}))
print("Sampled files:", len(sample))

In [ ]:
def segment_diagnostics(raw: pd.DataFrame) -> dict:
    ordered = raw.sort_values("timestamp", kind="stable").reset_index(drop=True)
    if len(ordered) < 2:
        return {
            "median_gap_s": np.nan,
            "p95_gap_s": np.nan,
            "max_gap_s_raw": np.nan,
            "p95_speed_kmh_raw": np.nan,
            "p99_speed_kmh_raw": np.nan,
            "max_speed_kmh_raw": np.nan,
            "raw_edges_gt_300_kmh": 0,
            "raw_edges_gt_1200_kmh": 0,
        }

    timestamp = pd.to_datetime(ordered["timestamp"], utc=True)
    dt_s = timestamp.diff().dt.total_seconds().to_numpy(dtype=float)[1:]

    lat = ordered["latitude"].to_numpy(dtype=float)
    lon = ordered["longitude"].to_numpy(dtype=float)
    distance_m = np.asarray(
        haversine_m(lat[:-1], lon[:-1], lat[1:], lon[1:]),
        dtype=float,
    )

    speed_kmh = np.full(dt_s.shape, np.nan, dtype=float)
    positive = dt_s > 0
    speed_kmh[positive] = distance_m[positive] / dt_s[positive] * 3.6

    finite_gap = dt_s[np.isfinite(dt_s)]
    finite_speed = speed_kmh[np.isfinite(speed_kmh)]

    return {
        "median_gap_s": float(np.median(finite_gap)) if finite_gap.size else np.nan,
        "p95_gap_s": float(np.quantile(finite_gap, 0.95)) if finite_gap.size else np.nan,
        "max_gap_s_raw": float(np.max(finite_gap)) if finite_gap.size else np.nan,
        "p95_speed_kmh_raw": float(np.quantile(finite_speed, 0.95)) if finite_speed.size else np.nan,
        "p99_speed_kmh_raw": float(np.quantile(finite_speed, 0.99)) if finite_speed.size else np.nan,
        "max_speed_kmh_raw": float(np.max(finite_speed)) if finite_speed.size else np.nan,
        "raw_edges_gt_300_kmh": int((finite_speed > 300).sum()),
        "raw_edges_gt_1200_kmh": int((finite_speed > 1200).sum()),
    }

def detect_on_cleaned(cleaned, *, radius_m, dwell_s):
    return detect_staypoints(
        cleaned,
        distance_threshold_m=radius_m,
        min_dwell_s=dwell_s,
    )

def summarize_file(user_id: str, path: Path) -> dict:
    raw = read_plt(path)[["timestamp", "latitude", "longitude"]]

    cleaned_by_gap = {}
    audit_baseline = None

    for gap in (120, 300, 600):
        if gap == BASELINE["max_gap_s"]:
            cleaned, audit = clean_trajectory_with_audit(
                raw,
                same_second_radius_m=BASELINE["same_second_radius_m"],
                max_gap_s=gap,
                hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
            )
            cleaned_by_gap[gap] = cleaned
            audit_baseline = audit
        else:
            cleaned_by_gap[gap] = clean_trajectory(
                raw,
                same_second_radius_m=BASELINE["same_second_radius_m"],
                max_gap_s=gap,
                hard_speed_guard_kmh=BASELINE["hard_speed_guard_kmh"],
            )

    row = {
        "user_id": user_id,
        "file": str(path),
        "n_raw": len(raw),
        **segment_diagnostics(raw),
    }

    # Gap sensitivity: radius/dwell frozen at baseline.
    for gap in (120, 300, 600):
        cleaned = cleaned_by_gap[gap]
        stays = detect_on_cleaned(
            cleaned,
            radius_m=BASELINE["distance_threshold_m"],
            dwell_s=BASELINE["min_dwell_s"],
        )
        row[f"gap_{gap}_sequences"] = (
            int(cleaned["sequence_id"].nunique()) if len(cleaned) else 0
        )
        row[f"gap_{gap}_stays"] = int(len(stays))

    baseline_cleaned = cleaned_by_gap[BASELINE["max_gap_s"]]

    # Radius sensitivity: same baseline-cleaned observations.
    for radius in (100, 200, 300):
        stays = detect_on_cleaned(
            baseline_cleaned,
            radius_m=radius,
            dwell_s=BASELINE["min_dwell_s"],
        )
        row[f"radius_{radius}_stays"] = int(len(stays))

    # Dwell sensitivity: same baseline-cleaned observations.
    for dwell in (600, 1200, 1800):
        stays = detect_on_cleaned(
            baseline_cleaned,
            radius_m=BASELINE["distance_threshold_m"],
            dwell_s=dwell,
        )
        row[f"dwell_{dwell}_stays"] = int(len(stays))

    row["baseline_sequences"] = (
        int(baseline_cleaned["sequence_id"].nunique())
        if len(baseline_cleaned)
        else 0
    )
    row["temporal_gap_boundaries"] = int(
        (audit_baseline["reason"] == "temporal_gap").sum()
    )
    row["hard_speed_boundaries"] = int(
        (audit_baseline["reason"] == "hard_speed_guard").sum()
    )

    return row

if CASE_SCAN_CACHE.exists():
    case_scan = pd.read_pickle(CASE_SCAN_CACHE)
    print("Loaded:", CASE_SCAN_CACHE)
else:
    rows = []
    t0 = perf_counter()

    for i, (user_id, path) in enumerate(sample, 1):
        rows.append(summarize_file(user_id, path))
        if i % 100 == 0:
            print(f"{i:,}/{len(sample):,} files | {(perf_counter() - t0)/60:.1f} min")

    case_scan = pd.DataFrame(rows)
    case_scan.to_pickle(CASE_SCAN_CACHE)
    print("Saved:", CASE_SCAN_CACHE)

display(case_scan.head())

## 2. Data context trước khi nhìn case

Đây là **context**, không phải rule để tự động suy ra threshold.

Ta nhìn hai thứ:

- distribution của `p95` sampling gap theo trajectory;
- tail của maximum segment speed.

Mục tiêu là hiểu scale của uncertainty trong raw data trước khi đi xuống từng journey cụ thể.

In [ ]:
gap_context = case_scan["p95_gap_s"].dropna()
if len(gap_context):
    clip_gap = gap_context.quantile(0.99)

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(gap_context.clip(upper=clip_gap), bins=60)

    for value, label in [
        (120, "120 s"),
        (300, "300 s baseline"),
        (600, "600 s"),
    ]:
        ax.axvline(value, linestyle="--", label=label)

    ax.set(
        title="Trajectory p95 sampling gaps in deterministic sample (clipped p99)",
        xlabel="p95 inter-observation gap (seconds)",
        ylabel="trajectory files",
    )
    ax.legend()
    plt.show()

speed_context = case_scan["max_speed_kmh_raw"].dropna()
speed_context = speed_context[speed_context > 0]

if len(speed_context):
    speed_context = case_scan["max_speed_kmh_raw"].dropna()
    speed_context = speed_context[speed_context > 0]
    
    if len(speed_context):
        lower = speed_context.min()
        upper = speed_context.max()
    
        bins = np.logspace(
            np.log10(lower),
            np.log10(upper),
            50,
        )
    
        fig, ax = plt.subplots(figsize=(9, 4))
    
        ax.hist(
            speed_context,
            bins=bins,
        )
    
        ax.axvline(
            BASELINE["hard_speed_guard_kmh"],
            linestyle="--",
            linewidth=1.5,
            label="1200 km/h hard guard",
        )
    
        ax.set_xscale("log")
    
        ax.set(
            title="Maximum raw segment speed per trajectory",
            xlabel="maximum segment speed (km/h, log scale)",
            ylabel="trajectory files",
        )
    
        ax.grid(alpha=0.2)
        ax.legend()
    
        plt.show()
    ax.set(
        title="Maximum raw segment speed per trajectory",
        xlabel="max segment speed (km/h, log scale)",
        ylabel="trajectory files",
    )
    ax.legend()
    plt.show()

**Sampling gap.** Phần lớn trajectory có `p95 inter-observation gap` khá nhỏ, nhưng vẫn có long tail, nghĩa là GeoLife không được sample với một interval cố định. Vì vậy `max_gap_s` được dùng như một **continuity policy**: gap càng lớn thì càng ít evidence để nối hai observations thành cùng sequence. Các mức `120 / 300 / 600 s` lần lượt thể hiện strict → baseline → permissive.

**Maximum segment speed.** Với mỗi trajectory, ta lấy tốc độ lớn nhất giữa hai GPS observations liên tiếp. Phần lớn trajectory maxima nằm thấp hơn nhiều so với `1200 km/h`, nhưng vẫn có một extreme tail vượt ngưỡng này. Vì vậy `1200 km/h` chỉ là **hard continuity guard** để cắt các jump cực đoan, không phải threshold phân loại phương tiện.

> Hai plot này chỉ cung cấp **data-level rationale**. Chúng không chứng minh `300 s` hay `1200 km/h` là tối ưu; quyết định baseline còn được kiểm tra tiếp bằng trajectory case studies và 27-config sensitivity.

Sampling gap là gì?

Giả sử trong một trajectory có các GPS observations:

Point A: 10:00:00
Point B: 10:00:05
Point C: 10:00:11
Point D: 10:00:40

Thì sampling gap là khoảng thời gian giữa hai GPS observations liên tiếp:

A → B = 5 giây
B → C = 6 giây
C → D = 29 giây

Tức đơn giản:

sampling_gap[i]
= timestamp[i] - timestamp[i-1]

Nó không phải thời gian user dừng lại. Nó chỉ nói:

GPS logger mất bao lâu mới ghi observation tiếp theo?

### Cách đọc data context

- `300 s` không mang nghĩa "sau 5 phút user đổi hành vi". Nó là **continuity policy**.
- `1200 km/h` không phải transport-mode cutoff. Nó là **extreme pathology guard**.

Histogram chỉ cho biết các candidate values nằm ở đâu so với behavior của data.
Rationale cuối phải đến từ trajectory cases + population sensitivity, không phải từ việc nhìn histogram rồi chọn một con số.

## 3. Chọn trajectory để làm case study

Mục tiêu của phần này không phải chọn trajectory mà baseline cho kết quả "đẹp nhất",
mà chọn các trajectory **nhạy với parameter** để nhìn rõ trade-off.

Ví dụ với `max_gap_s = 300`:

```text
120 s  → strict
300 s  → baseline
600 s  → permissive
````

Ta ưu tiên case mà kết quả thay đổi ở cả hai phía:

```text
120 s → 0 stays
300 s → 1 stay
600 s → 2 stays
```

Tương tự với radius và dwell.

Nếu không tìm được case thay đổi ở cả hai phía, notebook chọn case thay đổi ở ít nhất một phía.
Trong các case hợp lệ, trajectory ít raw GPS points hơn được ưu tiên để visualization dễ đọc.

> Case selection chỉ phục vụ giải thích behavior của threshold, không dùng để chứng minh baseline là optimal.

In [ ]:
def choose_readable_case(df, used_files, masks, min_points=20):
    for mask in masks:
        candidates = df.loc[
            mask
            & ~df["file"].isin(used_files)
            & (df["n_raw"] >= min_points)
        ].copy()

        if candidates.empty:
            continue

        candidates = candidates.sort_values(
            ["n_raw", "user_id", "file"],
            kind="stable",
        )

        row = candidates.iloc[0]
        used_files.add(row["file"])
        return row

    return None

used_files = set()

gap_case = choose_readable_case(
    case_scan,
    used_files,
    [
        (case_scan["gap_120_stays"] < case_scan["gap_300_stays"])
        & (case_scan["gap_300_stays"] < case_scan["gap_600_stays"])
        & (case_scan["gap_300_stays"] > 0),
        (case_scan["gap_120_stays"] < case_scan["gap_300_stays"])
        & (case_scan["gap_300_stays"] > 0),
        (case_scan["gap_300_stays"] < case_scan["gap_600_stays"])
        & (case_scan["gap_300_stays"] > 0),
        case_scan["temporal_gap_boundaries"] > 0,
    ],
)

radius_case = choose_readable_case(
    case_scan,
    used_files,
    [
        (case_scan["radius_100_stays"] < case_scan["radius_200_stays"])
        & (case_scan["radius_200_stays"] < case_scan["radius_300_stays"])
        & (case_scan["radius_200_stays"] > 0),
        (case_scan["radius_100_stays"] < case_scan["radius_200_stays"])
        & (case_scan["radius_200_stays"] > 0),
        (case_scan["radius_200_stays"] != case_scan["radius_300_stays"])
        & (case_scan["radius_200_stays"] > 0),
    ],
)

dwell_case = choose_readable_case(
    case_scan,
    used_files,
    [
        (case_scan["dwell_600_stays"] > case_scan["dwell_1200_stays"])
        & (case_scan["dwell_1200_stays"] > case_scan["dwell_1800_stays"])
        & (case_scan["dwell_1200_stays"] > 0),
        (case_scan["dwell_600_stays"] > case_scan["dwell_1200_stays"])
        & (case_scan["dwell_1200_stays"] > 0),
        (case_scan["dwell_1200_stays"] > case_scan["dwell_1800_stays"])
        & (case_scan["dwell_1200_stays"] > 0),
    ],
)

hard_speed_case = choose_readable_case(
    case_scan,
    used_files,
    [
        case_scan["hard_speed_boundaries"] > 0,
    ],
)

selected_cases = pd.DataFrame(
    [
        {"case": "gap", **(gap_case.to_dict() if gap_case is not None else {})},
        {"case": "radius", **(radius_case.to_dict() if radius_case is not None else {})},
        {"case": "dwell", **(dwell_case.to_dict() if dwell_case is not None else {})},
        {"case": "hard_speed", **(hard_speed_case.to_dict() if hard_speed_case is not None else {})},
    ]
)

columns_to_show = [
    "case", "user_id", "file", "n_raw",
    "gap_120_sequences", "gap_300_sequences", "gap_600_sequences",
    "gap_120_stays", "gap_300_stays", "gap_600_stays",
    "radius_100_stays", "radius_200_stays", "radius_300_stays",
    "dwell_600_stays", "dwell_1200_stays", "dwell_1800_stays",
    "temporal_gap_boundaries", "hard_speed_boundaries",
]
display(selected_cases.reindex(columns=columns_to_show))

## 4. Mentor-facing interactive maps

Journey visualization được export thành **Folium HTML** riêng và dùng **MapTiler Streets** làm basemap.

### One-time setup

Tạo một MapTiler API key rồi trong **cell tạm, không commit** chạy:

```python
import os
os.environ["MAPTILER_API_KEY"] = "YOUR_KEY"
```

Sau đó rerun helper cell này và các Case A/B/C/D export cells.

Map hỗ trợ zoom/pan/fullscreen, animated sequence path, config layers, stay popup/cluster và radius circles.

> **Không commit HTML output**: key được nhúng trong tile URL của file HTML và HTML cũng chứa precise GPS coordinates.

In [ ]:
# Interactive mentor-demo maps.
# The HTML is exported and opened in a normal browser tab.
# OpenStreetMap tiles are requested by the browser, not by Modal's notebook iframe.
try:
    import folium
    from folium import plugins
except ImportError:
    print("Installing folium...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "folium>=0.15,<1"],
        check=True,
    )
    import folium
    from folium import plugins

EXPORT_DIR = Path(
    os.environ.get(
        "GEOLIFE_MAP_EXPORT_DIR",
        "/mnt/geolife-data/exports/trajectory_case_maps",
    )
)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# MapTiler key is read from the runtime environment.
# Do not hard-code it in a public notebook.
MAPTILER_API_KEY = os.environ.get("MAPTILER_API_KEY", "").strip()


def require_maptiler_key():
    if not MAPTILER_API_KEY:
        raise RuntimeError(
            "MAPTILER_API_KEY is not set. In a temporary/private cell run: "
            'os.environ["MAPTILER_API_KEY"] = "YOUR_MAPTILER_KEY" '
            "then rerun this helper cell."
        )
    return MAPTILER_API_KEY


def run_config(
    path,
    *,
    gap_s=300,
    radius_m=200,
    dwell_s=1200,
    speed_guard_kmh=None,
):
    raw = read_plt(Path(path))[["timestamp", "latitude", "longitude"]]

    if speed_guard_kmh is None:
        speed_guard_kmh = BASELINE["hard_speed_guard_kmh"]

    cleaned = clean_trajectory(
        raw,
        same_second_radius_m=BASELINE["same_second_radius_m"],
        max_gap_s=gap_s,
        hard_speed_guard_kmh=speed_guard_kmh,
    )

    stays = detect_staypoints(
        cleaned,
        distance_threshold_m=radius_m,
        min_dwell_s=dwell_s,
    )

    return raw, cleaned, stays


def comparison_table(path, *, axis_name, values, base_config=None):
    rows = []

    base_config = dict(base_config or {})

    for value in values:
        kwargs = {
            "gap_s": BASELINE["max_gap_s"],
            "radius_m": BASELINE["distance_threshold_m"],
            "dwell_s": BASELINE["min_dwell_s"],
            "speed_guard_kmh": BASELINE["hard_speed_guard_kmh"],
        }
        kwargs.update(base_config)

        if axis_name == "gap_s":
            kwargs["gap_s"] = value
        elif axis_name == "radius_m":
            kwargs["radius_m"] = value
        elif axis_name == "dwell_s":
            kwargs["dwell_s"] = value
        elif axis_name == "speed_guard_kmh":
            kwargs["speed_guard_kmh"] = value
        else:
            raise ValueError(axis_name)

        _, cleaned, stays = run_config(path, **kwargs)

        rows.append(
            {
                axis_name: value,
                "sequences": int(cleaned["sequence_id"].nunique()) if len(cleaned) else 0,
                "stays": int(len(stays)),
                "stay_duration_min": (
                    [round(float(v) / 60.0, 1) for v in stays["duration_s"]]
                    if len(stays)
                    else []
                ),
            }
        )

    return pd.DataFrame(rows)


def _stay_members(cleaned: pd.DataFrame, stay):
    return cleaned.loc[
        (cleaned["sequence_id"] == stay.sequence_id)
        & (cleaned["timestamp"] >= stay.arrival_time)
        & (cleaned["timestamp"] <= stay.departure_time)
    ]


def _add_stay_marker(group, stay, stay_idx, config_name, *, color="red"):
    popup_html = (
        f"<b>Stay {stay_idx}</b><br>"
        f"config: {config_name}<br>"
        f"arrival: {stay.arrival_time}<br>"
        f"departure: {stay.departure_time}<br>"
        f"duration: {stay.duration_s/60:.1f} min<br>"
        f"points: {stay.n_points}<br>"
        f"sequence: {stay.sequence_id}"
    )

    folium.Marker(
        [stay.latitude, stay.longitude],
        tooltip=f"Stay {stay_idx} — {stay.duration_s/60:.1f} min",
        popup=folium.Popup(popup_html, max_width=380),
        icon=folium.DivIcon(
            icon_size=(30, 30),
            icon_anchor=(15, 15),
            html=(
                '<div style="'
                'width:30px;height:30px;line-height:30px;'
                'border-radius:15px;text-align:center;'
                'background:#d73027;color:white;'
                'font-weight:bold;border:2px solid white;'
                'box-shadow:0 0 4px #555;">'
                f'{stay_idx}</div>'
            ),
        ),
    ).add_to(group)


def build_case_map(
    path,
    configs,
    *,
    title,
    draw_radius=False,
    boundary_reason=None,
):
    runs = []
    all_coords = []

    for config in configs:
        raw, cleaned, stays = run_config(
            path,
            gap_s=config["gap_s"],
            radius_m=config["radius_m"],
            dwell_s=config["dwell_s"],
            speed_guard_kmh=config.get(
                "speed_guard_kmh",
                BASELINE["hard_speed_guard_kmh"],
            ),
        )
        runs.append((config, raw, cleaned, stays))

        if len(cleaned):
            all_coords.extend(
                cleaned[["latitude", "longitude"]].to_numpy(dtype=float).tolist()
            )

    if not all_coords:
        raise ValueError("No cleaned coordinates available for map.")

    default_idx = next(
        (i for i, (config, *_rest) in enumerate(runs) if config.get("show", False)),
        len(runs) // 2,
    )
    default_config, default_raw, default_cleaned, default_stays = runs[default_idx]

    center = [
        float(default_cleaned["latitude"].median()),
        float(default_cleaned["longitude"].median()),
    ]

    # MapTiler raster basemap (no direct use of OSM volunteer tile servers).
    key = require_maptiler_key()
    maptiler_url = (
        "https://api.maptiler.com/maps/streets-v4/256/"
        "{z}/{x}/{y}.png?key=" + key
    )
    maptiler_attr = (
        '&copy; <a href="https://www.maptiler.com/copyright/">MapTiler</a> '
        '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap contributors</a>'
    )

    m = folium.Map(
        location=center,
        zoom_start=13,
        tiles=None,
        control_scale=True,
        prefer_canvas=True,
    )

    folium.TileLayer(
        tiles=maptiler_url,
        attr=maptiler_attr,
        name="MapTiler Streets",
        min_zoom=1,
        max_zoom=19,
        overlay=False,
        control=False,
        show=True,
    ).add_to(m)

    plugins.Fullscreen(position="topleft").add_to(m)

    # Explicit provider for MiniMap too, avoiding fallback to tile.openstreetmap.org.
    mini_tiles = folium.TileLayer(
        tiles=maptiler_url,
        attr=maptiler_attr,
        min_zoom=1,
        max_zoom=19,
        overlay=False,
        control=False,
        show=True,
    )
    plugins.MiniMap(
        tile_layer=mini_tiles,
        toggle_display=True,
        minimized=True,
    ).add_to(m)
    plugins.MeasureControl(
        position="topleft",
        primary_length_unit="meters",
        secondary_length_unit="kilometers",
    ).add_to(m)
    plugins.MousePosition(
        position="bottomright",
        separator=" | ",
        prefix="lat/lon:",
        num_digits=5,
    ).add_to(m)

    # Global first/last observed point from baseline/default config.
    if len(default_cleaned):
        orient = folium.FeatureGroup(name="First / last observed point", show=True)

        first = default_cleaned.iloc[0]
        last = default_cleaned.iloc[-1]

        folium.Marker(
            [first.latitude, first.longitude],
            tooltip="First observed point",
            popup=f"<b>First observed point</b><br>{first.timestamp}",
            icon=folium.Icon(color="green", icon="play"),
        ).add_to(orient)

        folium.Marker(
            [last.latitude, last.longitude],
            tooltip="Last observed point",
            popup=f"<b>Last observed point</b><br>{last.timestamp}",
            icon=folium.Icon(color="black", icon="stop"),
        ).add_to(orient)

        orient.add_to(m)

    config_colors = ["#7b3294", "#008837", "#e66101", "#0571b0"]
    config_layers = []

    for config_idx, (config, raw, cleaned, stays) in enumerate(runs):
        color = config_colors[config_idx % len(config_colors)]
        group = folium.FeatureGroup(
            name=config["name"],
            show=bool(config.get("show", False)),
        )

        # Draw sequences separately so cleaning boundaries remain visually honest.
        for sequence_id, sequence in cleaned.groupby("sequence_id", sort=False):
            coords = sequence[["latitude", "longitude"]].to_numpy(dtype=float).tolist()
            if not coords:
                continue

            start_time = sequence["timestamp"].iloc[0]
            end_time = sequence["timestamp"].iloc[-1]
            tooltip = (
                f"Sequence {sequence_id} | points={len(sequence)} | "
                f"{start_time} → {end_time}"
            )

            if len(coords) >= 2:
                folium.PolyLine(
                    coords,
                    color=color,
                    weight=5,
                    opacity=0.45,
                    tooltip=tooltip,
                ).add_to(group)

                plugins.AntPath(
                    coords,
                    color=color,
                    pulse_color="#ffffff",
                    delay=900,
                    dash_array=[10, 18],
                    weight=3,
                    opacity=0.9,
                ).add_to(group)
            else:
                folium.CircleMarker(
                    coords[0],
                    radius=3,
                    color=color,
                    fill=True,
                    fill_opacity=0.9,
                    tooltip=tooltip,
                ).add_to(group)

        # Mark cleaning boundary events when relevant.
        if boundary_reason and "boundary_before_reason" in cleaned.columns:
            hits = cleaned.loc[
                cleaned["boundary_before_reason"] == boundary_reason
            ]
            for row in hits.itertuples(index=False):
                folium.CircleMarker(
                    [row.latitude, row.longitude],
                    radius=7,
                    color="#d73027",
                    weight=2,
                    fill=True,
                    fill_color="#fee08b",
                    fill_opacity=0.95,
                    tooltip=f"{boundary_reason} @ {row.timestamp}",
                    popup=(
                        f"<b>{boundary_reason}</b><br>"
                        f"timestamp: {row.timestamp}<br>"
                        f"new sequence: {row.sequence_id}"
                    ),
                ).add_to(group)

        # Cluster makes same/nearby stays easy to inspect.
        cluster = plugins.MarkerCluster(
            options={
                "spiderfyOnMaxZoom": True,
                "showCoverageOnHover": False,
                "zoomToBoundsOnClick": True,
            }
        )
        cluster.add_to(group)

        for stay_idx, stay in enumerate(stays.itertuples(index=False), start=1):
            members = _stay_members(cleaned, stay)

            _add_stay_marker(
                cluster,
                stay,
                stay_idx,
                config["name"],
            )

            # Small stay-member points.
            for member in members.itertuples(index=False):
                folium.CircleMarker(
                    [member.latitude, member.longitude],
                    radius=3,
                    color="#d73027",
                    weight=1,
                    fill=True,
                    fill_color="#fc8d59",
                    fill_opacity=0.85,
                    tooltip=f"Stay {stay_idx} member @ {member.timestamp}",
                ).add_to(group)

            # Radius case: circle around actual arrival anchor.
            if draw_radius and len(members):
                anchor = members.iloc[0]

                folium.Circle(
                    [anchor.latitude, anchor.longitude],
                    radius=float(config["radius_m"]),
                    color="#d73027",
                    weight=2,
                    dash_array="6,6",
                    fill=False,
                    tooltip=f"Stay {stay_idx} radius = {config['radius_m']} m",
                ).add_to(group)

                folium.CircleMarker(
                    [anchor.latitude, anchor.longitude],
                    radius=5,
                    color="#313695",
                    weight=2,
                    fill=True,
                    fill_color="#74add1",
                    fill_opacity=1,
                    tooltip=f"Stay {stay_idx} arrival anchor",
                ).add_to(group)

        group.add_to(m)
        config_layers.append(group)

    # Normal layer control is the most robust in exported HTML.
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds(all_coords)

    # Small mentor-facing explanation box.
    explanation = f"""
    <div style="
        position: fixed;
        top: 10px;
        left: 50%;
        transform: translateX(-50%);
        z-index: 9999;
        background: rgba(255,255,255,0.94);
        padding: 8px 14px;
        border: 1px solid #bbb;
        border-radius: 8px;
        box-shadow: 0 1px 5px rgba(0,0,0,0.25);
        font-family: sans-serif;
        max-width: 72%;
        text-align: center;
    ">
      <b>{title}</b><br>
      <span style="font-size:12px">
        Animated line = observation order inside each sequence ·
        red numbered marker = emitted stay · click markers for details
      </span>
    </div>
    """
    m.get_root().html.add_child(folium.Element(explanation))

    return m


def export_case_map(map_obj, filename):
    path = EXPORT_DIR / filename
    map_obj.save(str(path))
    print(f"Saved interactive map: {path}")
    print("Open/download this HTML in a normal browser tab for mentor demo.")
    print("Security: HTML contains the MapTiler key in tile URLs; do not commit/share publicly.")
    return path


def plot_time_gaps(path):
    raw = (
        read_plt(Path(path))[["timestamp", "latitude", "longitude"]]
        .sort_values("timestamp", kind="stable")
        .reset_index(drop=True)
    )

    timestamp = pd.to_datetime(raw["timestamp"], utc=True)
    gaps = timestamp.diff().dt.total_seconds()

    gap_df = pd.DataFrame({
        "transition": np.arange(1, len(raw)),
        "gap_s": gaps.iloc[1:].to_numpy(),
        "from_time": timestamp.iloc[:-1].to_numpy(),
        "to_time": timestamp.iloc[1:].to_numpy(),
    })

    y_cap = 900.0
    gap_df["display_gap_s"] = gap_df["gap_s"].clip(upper=y_cap)

    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.scatter(gap_df["transition"], gap_df["display_gap_s"], s=24, zorder=3)
    ax.vlines(
        gap_df["transition"], 0, gap_df["display_gap_s"],
        alpha=0.25, linewidth=0.8, zorder=1,
    )

    for threshold, label in [
        (120, "120 s — strict"),
        (300, "300 s — baseline"),
        (600, "600 s — permissive"),
    ]:
        ax.axhline(
            threshold,
            linestyle="--",
            linewidth=1.2,
            label=label,
            zorder=2,
        )

    extreme = gap_df[gap_df["gap_s"] > y_cap]
    for row in extreme.itertuples(index=False):
        ax.annotate(
            f"{row.gap_s:.0f}s\n(clipped)",
            xy=(row.transition, y_cap),
            xytext=(0, -35),
            textcoords="offset points",
            ha="center",
            va="top",
            arrowprops={"arrowstyle": "->"},
        )

    ax.set(
        title="Inter-observation gaps — threshold-focused view",
        xlabel="consecutive observation transition",
        ylabel="gap (seconds)",
        ylim=(0, y_cap + 50),
    )
    ax.grid(alpha=0.2)
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0,
    )
    fig.tight_layout(rect=(0, 0, 0.80, 1))
    plt.show()

    summary = pd.DataFrame({
        "threshold_s": [120, 300, 600],
        "gaps_above_threshold": [
            int((gap_df["gap_s"] > 120).sum()),
            int((gap_df["gap_s"] > 300).sum()),
            int((gap_df["gap_s"] > 600).sum()),
        ],
    })
    display(summary)

    if len(extreme):
        print("Long gaps clipped in the figure:")
        display(
            extreme[
                ["transition", "from_time", "to_time", "gap_s"]
            ].sort_values("gap_s", ascending=False)
        )

## 4.1 Cách đọc kết luận của 02a

Mỗi Case trong 02a trả lời hai câu khác nhau:

```text
1. Khi thay parameter này, behavior thay đổi như thế nào?
2. Case cụ thể này support được kết luận nào?
```

02a **không** trả lời trực tiếp:

```text
"Giá trị nào phải trở thành production default?"
```

Lý do: một trajectory nhạy với parameter là rất hữu ích để giải thích mechanics, nhưng không đại diện cho toàn bộ 18,670 trajectory files.

Vì vậy:

- Case A/B/C/D có thể nói một giá trị **hợp lý / quá strict / tạo behavior khác** trên case cụ thể;
- nếu hai config đều hợp lệ theo detector và không có ground truth phân biệt, notebook phải ghi **"chưa đủ evidence để chọn"**;
- production freeze chuyển sang notebook **02b**, nơi chạy population-level sensitivity trên nhiều users/configs.

> Đặc biệt: **nhiều stay hơn không tự động tốt hơn hoặc xấu hơn**.


In [ ]:

# Deep-dive diagnostics and automatic case conclusions.
# These helpers do not redefine the staypoint algorithm; they only inspect its behavior.

CASE_FINDINGS = {}


def consecutive_edge_table(path):
    raw = (
        read_plt(Path(path))[["timestamp", "latitude", "longitude"]]
        .sort_values("timestamp", kind="stable")
        .reset_index(drop=True)
    )
    if len(raw) < 2:
        return pd.DataFrame()

    ts = pd.to_datetime(raw["timestamp"], utc=True)
    lat = raw["latitude"].to_numpy(dtype=float)
    lon = raw["longitude"].to_numpy(dtype=float)

    dt_s = ts.diff().dt.total_seconds().to_numpy(dtype=float)[1:]
    distance_m = np.asarray(
        haversine_m(lat[:-1], lon[:-1], lat[1:], lon[1:]),
        dtype=float,
    )

    speed_kmh = np.full(len(dt_s), np.nan, dtype=float)
    positive = dt_s > 0
    speed_kmh[positive] = distance_m[positive] / dt_s[positive] * 3.6

    return pd.DataFrame({
        "from_time": ts.iloc[:-1].to_numpy(),
        "to_time": ts.iloc[1:].to_numpy(),
        "from_lat": lat[:-1],
        "from_lon": lon[:-1],
        "to_lat": lat[1:],
        "to_lon": lon[1:],
        "dt_s": dt_s,
        "distance_m": distance_m,
        "speed_kmh": speed_kmh,
    })


def gap_bridge_diagnostics(path, *, lower_s=300, upper_s=600):
    edges = consecutive_edge_table(path)
    if edges.empty:
        return edges

    bridges = edges.loc[
        (edges["dt_s"] > lower_s)
        & (edges["dt_s"] <= upper_s)
    ].copy()

    if bridges.empty:
        return bridges

    bridges["distance_km"] = bridges["distance_m"] / 1000.0
    bridges["under_hard_speed_guard"] = (
        bridges["speed_kmh"] <= BASELINE["hard_speed_guard_kmh"]
    )
    return bridges[
        [
            "from_time", "to_time", "dt_s",
            "distance_m", "distance_km", "speed_kmh",
            "under_hard_speed_guard",
        ]
    ].sort_values("dt_s", ascending=False)


def _distance_series_from_point(lat0, lon0, lat, lon):
    return np.asarray(
        haversine_m(
            np.full(len(lat), float(lat0)),
            np.full(len(lon), float(lon0)),
            np.asarray(lat, dtype=float),
            np.asarray(lon, dtype=float),
        ),
        dtype=float,
    )


def stay_spatial_diagnostics(
    path,
    *,
    radius_values=(100, 200, 300),
    gap_s=300,
    dwell_s=1200,
    speed_guard_kmh=1200,
):
    rows = []

    for radius_m in radius_values:
        _, cleaned, stays = run_config(
            path,
            gap_s=gap_s,
            radius_m=radius_m,
            dwell_s=dwell_s,
            speed_guard_kmh=speed_guard_kmh,
        )

        for stay_id, stay in enumerate(stays.itertuples(index=False), start=1):
            members = _stay_members(cleaned, stay).sort_values(
                "timestamp", kind="stable"
            )
            if members.empty:
                continue

            lat = members["latitude"].to_numpy(dtype=float)
            lon = members["longitude"].to_numpy(dtype=float)
            ts = pd.to_datetime(members["timestamp"], utc=True)

            anchor_lat = float(lat[0])
            anchor_lon = float(lon[0])
            anchor_dist = _distance_series_from_point(
                anchor_lat, anchor_lon, lat, lon
            )

            center_lat = float(np.median(lat))
            center_lon = float(np.median(lon))
            center_dist = _distance_series_from_point(
                center_lat, center_lon, lat, lon
            )

            if len(members) >= 2:
                step_dist = np.asarray(
                    haversine_m(lat[:-1], lon[:-1], lat[1:], lon[1:]),
                    dtype=float,
                )
                start_end_m = float(
                    np.asarray(
                        haversine_m(
                            np.array([lat[0]]), np.array([lon[0]]),
                            np.array([lat[-1]]), np.array([lon[-1]]),
                        )
                    ).reshape(-1)[0]
                )
                internal_gap_s = (
                    ts.diff().dt.total_seconds().dropna().to_numpy(dtype=float)
                )
                max_internal_gap_s = (
                    float(np.max(internal_gap_s))
                    if len(internal_gap_s)
                    else np.nan
                )
                path_length_m = float(np.sum(step_dist))
            else:
                start_end_m = 0.0
                max_internal_gap_s = np.nan
                path_length_m = 0.0

            rows.append({
                "radius_m": radius_m,
                "stay_id": stay_id,
                "arrival": stay.arrival_time,
                "departure": stay.departure_time,
                "duration_min": round(float(stay.duration_s) / 60.0, 2),
                "n_points": int(stay.n_points),
                "sequence_id": int(stay.sequence_id),
                "p95_anchor_distance_m": float(np.quantile(anchor_dist, 0.95)),
                "max_anchor_distance_m": float(np.max(anchor_dist)),
                "max_center_distance_m": float(np.max(center_dist)),
                "start_end_distance_m": start_end_m,
                "path_length_m": path_length_m,
                "max_internal_gap_s": max_internal_gap_s,
                "radius_utilization": (
                    float(np.max(anchor_dist)) / float(radius_m)
                    if radius_m > 0 else np.nan
                ),
            })

    return pd.DataFrame(rows)


def dwell_episode_diagnostics(
    path,
    *,
    dwell_values=(600, 1200, 1800),
    gap_s=300,
    radius_m=200,
    speed_guard_kmh=1200,
):
    rows = []

    for dwell_s in dwell_values:
        _, cleaned, stays = run_config(
            path,
            gap_s=gap_s,
            radius_m=radius_m,
            dwell_s=dwell_s,
            speed_guard_kmh=speed_guard_kmh,
        )

        for stay_id, stay in enumerate(stays.itertuples(index=False), start=1):
            rows.append({
                "dwell_s": dwell_s,
                "dwell_min": dwell_s / 60.0,
                "stay_id": stay_id,
                "arrival": stay.arrival_time,
                "departure": stay.departure_time,
                "duration_min": round(float(stay.duration_s) / 60.0, 2),
                "n_points": int(stay.n_points),
                "sequence_id": int(stay.sequence_id),
            })

    return pd.DataFrame(rows)


def speed_guard_comparison(
    path,
    *,
    speed_values=(300.0, 1200.0, 1e9),
    gap_s=300,
    radius_m=200,
    dwell_s=1200,
):
    raw = read_plt(Path(path))[["timestamp", "latitude", "longitude"]]
    rows = []

    for guard in speed_values:
        cleaned, audit = clean_trajectory_with_audit(
            raw,
            same_second_radius_m=BASELINE["same_second_radius_m"],
            max_gap_s=gap_s,
            hard_speed_guard_kmh=guard,
        )
        stays = detect_staypoints(
            cleaned,
            distance_threshold_m=radius_m,
            min_dwell_s=dwell_s,
        )

        rows.append({
            "speed_guard_kmh": guard,
            "label": (
                "guard off"
                if guard >= 1e8
                else f"{guard:.0f} km/h"
            ),
            "sequences": (
                int(cleaned["sequence_id"].nunique())
                if len(cleaned) else 0
            ),
            "hard_speed_boundaries": int(
                (audit["reason"] == "hard_speed_guard").sum()
            ),
            "temporal_gap_boundaries": int(
                (audit["reason"] == "temporal_gap").sum()
            ),
            "stays": int(len(stays)),
        })

    return pd.DataFrame(rows)


def hard_speed_event_diagnostics(path, *, threshold_kmh=1200.0):
    raw = (
        read_plt(Path(path))[["timestamp", "latitude", "longitude"]]
        .sort_values("timestamp", kind="stable")
        .reset_index(drop=True)
    )
    cleaned, audit = clean_trajectory_with_audit(
        raw,
        same_second_radius_m=BASELINE["same_second_radius_m"],
        max_gap_s=BASELINE["max_gap_s"],
        hard_speed_guard_kmh=threshold_kmh,
    )

    hits = audit.loc[audit["reason"] == "hard_speed_guard"].copy()
    if hits.empty:
        return pd.DataFrame()

    raw_ts = pd.to_datetime(raw["timestamp"], utc=True)
    rows = []

    for hit in hits.itertuples(index=False):
        hit_ts = pd.to_datetime(hit.timestamp, utc=True)
        current_candidates = raw.index[raw_ts == hit_ts].tolist()
        if not current_candidates:
            continue

        current_idx = current_candidates[0]
        prior_candidates = raw.index[raw_ts < hit_ts].tolist()
        if not prior_candidates:
            continue

        previous_idx = prior_candidates[-1]
        prev = raw.loc[previous_idx]
        cur = raw.loc[current_idx]

        dt_s = (
            pd.to_datetime(cur["timestamp"], utc=True)
            - pd.to_datetime(prev["timestamp"], utc=True)
        ).total_seconds()

        distance_m = float(
            np.asarray(
                haversine_m(
                    np.array([float(prev["latitude"])]),
                    np.array([float(prev["longitude"])]),
                    np.array([float(cur["latitude"])]),
                    np.array([float(cur["longitude"])]),
                )
            ).reshape(-1)[0]
        )
        speed_kmh = (
            distance_m / dt_s * 3.6
            if dt_s > 0 else np.inf
        )

        rows.append({
            "from_time": pd.to_datetime(prev["timestamp"], utc=True),
            "to_time": pd.to_datetime(cur["timestamp"], utc=True),
            "dt_s": dt_s,
            "distance_m": distance_m,
            "distance_km": distance_m / 1000.0,
            "implied_speed_kmh": speed_kmh,
            "guard_kmh": threshold_kmh,
            "times_over_guard": (
                speed_kmh / threshold_kmh
                if threshold_kmh > 0 else np.nan
            ),
        })

    return pd.DataFrame(rows)


def population_sensitivity_table(df, prefix, values, baseline_value):
    baseline_col = f"{prefix}_{baseline_value}_stays"
    base = df[baseline_col].fillna(0)
    rows = []

    for value in values:
        col = f"{prefix}_{value}_stays"
        current = df[col].fillna(0)
        rows.append({
            "value": value,
            "total_stays": int(current.sum()),
            "trajectories_with_stay": int((current > 0).sum()),
            "trajectories_changed_vs_baseline": int((current != base).sum()),
            "delta_total_stays_vs_baseline": int(current.sum() - base.sum()),
        })

    return pd.DataFrame(rows)



def adjacent_stay_pair_diagnostics(
    path,
    *,
    radius_m,
    gap_s=300,
    dwell_s=1200,
    speed_guard_kmh=1200,
):
    _, cleaned, stays = run_config(
        path,
        gap_s=gap_s,
        radius_m=radius_m,
        dwell_s=dwell_s,
        speed_guard_kmh=speed_guard_kmh,
    )
    if len(stays) < 2:
        return pd.DataFrame()

    ordered = stays.sort_values(
        ["arrival_time", "departure_time"],
        kind="stable",
    ).reset_index(drop=True)

    rows = []
    for i in range(len(ordered) - 1):
        a = ordered.iloc[i]
        b = ordered.iloc[i + 1]
        center_distance_m = float(
            np.asarray(
                haversine_m(
                    np.array([float(a["latitude"])]),
                    np.array([float(a["longitude"])]),
                    np.array([float(b["latitude"])]),
                    np.array([float(b["longitude"])]),
                )
            ).reshape(-1)[0]
        )
        time_gap_s = (
            pd.to_datetime(b["arrival_time"], utc=True)
            - pd.to_datetime(a["departure_time"], utc=True)
        ).total_seconds()

        rows.append({
            "radius_m": radius_m,
            "stay_a": i + 1,
            "stay_b": i + 2,
            "stay_a_duration_min": float(a["duration_s"]) / 60.0,
            "stay_b_duration_min": float(b["duration_s"]) / 60.0,
            "center_distance_m": center_distance_m,
            "time_gap_s": float(time_gap_s),
            "both_meet_dwell_rule": (
                float(a["duration_s"]) >= dwell_s
                and float(b["duration_s"]) >= dwell_s
            ),
        })

    return pd.DataFrame(rows)


def _emit_case_finding(case_name, *, finding, supports, cannot_decide):
    CASE_FINDINGS[case_name] = {
        "finding": finding,
        "supports": supports,
        "cannot_decide": cannot_decide,
    }
    display(
        Markdown(
            f"### Kết luận {case_name}\n\n"
            f"**Finding.** {finding}\n\n"
            f"**Case này support:** {supports}\n\n"
            f"**Case này chưa thể kết luận:** {cannot_decide}"
        )
    )


def analyze_gap(path):
    comp = comparison_table(
        path, axis_name="gap_s", values=(120, 300, 600)
    )
    bridges = gap_bridge_diagnostics(path, lower_s=300, upper_s=600)

    seq300 = int(comp.loc[comp["gap_s"] == 300, "sequences"].iloc[0])
    seq600 = int(comp.loc[comp["gap_s"] == 600, "sequences"].iloc[0])
    stay300 = int(comp.loc[comp["gap_s"] == 300, "stays"].iloc[0])
    stay600 = int(comp.loc[comp["gap_s"] == 600, "stays"].iloc[0])

    unsafe_bridges = (
        int((~bridges["under_hard_speed_guard"]).sum())
        if len(bridges) else 0
    )
    max_bridge_speed = (
        float(bridges["speed_kmh"].max())
        if len(bridges) else np.nan
    )

    if seq600 < seq300 and unsafe_bridges == 0:
        finding = (
            f"`600s` giảm fragmentation trên case này: sequence **{seq300} → {seq600}**, "
            f"stay **{stay300} → {stay600}**. "
            + (
                f"Các bridge `300–600s` có implied speed tối đa "
                f"**{max_bridge_speed:.1f} km/h**, không vượt hard-speed guard."
                if np.isfinite(max_bridge_speed)
                else "Không có bridge mới trong vùng `300–600s`."
            )
        )
        supports = (
            "`600s` là một **continuity candidate hợp lý cho trajectory này**; "
            "các đoạn được nối thêm không bị diagnostic speed hiện tại đánh dấu là bất thường."
        )
    else:
        finding = (
            f"Sequence `300s → 600s` là **{seq300} → {seq600}**, "
            f"và có **{unsafe_bridges}** bridge mới vượt hard-speed guard."
        )
        supports = (
            "Case này không cho evidence sạch để nói `600s` cải thiện continuity "
            "mà vẫn giữ spatial plausibility."
        )

    cannot_decide = (
        "Không được dùng một selected trajectory để tự đổi production baseline `300s → 600s`. "
        "Việc freeze gap phải dựa vào population-level sensitivity ở 02b."
    )

    return comp, bridges, finding, supports, cannot_decide


def analyze_radius(path, *, gap_s=300):
    comp = comparison_table(
        path,
        axis_name="radius_m",
        values=(100, 200, 300),
        base_config={"gap_s": gap_s},
    )
    diag = stay_spatial_diagnostics(path, gap_s=gap_s)
    pair_diag = adjacent_stay_pair_diagnostics(
        path,
        radius_m=300,
        gap_s=gap_s,
        dwell_s=int(BASELINE["min_dwell_s"]),
        speed_guard_kmh=float(BASELINE["hard_speed_guard_kmh"]),
    )

    counts = {
        int(row.radius_m): int(row.stays)
        for row in comp.itertuples(index=False)
    }

    finding = (
        f"Stay count thay đổi theo radius: "
        f"`100m={counts[100]}`, `200m={counts[200]}`, `300m={counts[300]}`. "
    )

    if counts[100] < counts[200]:
        finding += "`100m` restrictive hơn và bỏ ít nhất một stay mà `200m` giữ được. "
    if counts[300] > counts[200]:
        finding += (
            "`300m` emit thêm stay so với `200m`; stay bổ sung phải được xem là "
            "một output hợp lệ theo detector nếu nó vẫn thỏa dwell rule, "
            "không được tự gắn nhãn false positive. "
        )

    if len(pair_diag):
        pair = pair_diag.iloc[0]
        finding += (
            f"Ở `300m`, hai stay liên tiếp đầu tiên có center distance khoảng "
            f"**{pair['center_distance_m']:.0f} m**, time gap khoảng "
            f"**{pair['time_gap_s']:.0f} s**, và "
            f"{'cả hai đều' if pair['both_meet_dwell_rule'] else 'không phải cả hai'} "
            "thỏa dwell 20 phút."
        )

    supports = (
        "Case này cho biết radius càng lớn thì detector permissive hơn; "
        "`100m` có thể quá strict đối với trajectory này."
    )

    if counts[200] != counts[300]:
        cannot_decide = (
            "Case này **không đủ evidence để chọn `200m` hay `300m`**. "
            "Số stay khác nhau không tự cho biết stay nào đúng/sai; cần population/user-level "
            "evidence ở 02b hoặc ground truth phù hợp."
        )
    else:
        cannot_decide = (
            "Việc hai threshold cho cùng số stay trên một case không chứng minh chúng tương đương "
            "trên toàn dataset; final radius vẫn phải freeze ở 02b."
        )

    return comp, diag, pair_diag, finding, supports, cannot_decide


def analyze_dwell(path, *, gap_s=300, radius_m=200):
    comp = comparison_table(
        path,
        axis_name="dwell_s",
        values=(600, 1200, 1800),
        base_config={"gap_s": gap_s, "radius_m": radius_m},
    )
    diag = dwell_episode_diagnostics(
        path, gap_s=gap_s, radius_m=radius_m
    )

    counts = {
        int(row.dwell_s): int(row.stays)
        for row in comp.itertuples(index=False)
    }
    d10 = (
        diag.loc[diag["dwell_s"] == 600, "duration_min"].round(1).tolist()
        if len(diag) else []
    )
    d20 = (
        diag.loc[diag["dwell_s"] == 1200, "duration_min"].round(1).tolist()
        if len(diag) else []
    )
    d30 = (
        diag.loc[diag["dwell_s"] == 1800, "duration_min"].round(1).tolist()
        if len(diag) else []
    )

    finding = (
        f"`10min` emit **{counts[600]}** stay {d10}; "
        f"`20min` emit **{counts[1200]}** stay {d20}; "
        f"`30min` emit **{counts[1800]}** stay {d30}."
    )
    supports = (
        "Case này minh họa trực tiếp dwell threshold đang filter các episode theo duration. "
        "Nếu product policy muốn 'sustained presence' từ 20 phút trở lên, `20min` là policy "
        "nhất quán với định nghĩa đó trên case này."
    )
    cannot_decide = (
        "Không có ground truth để gọi các stay `10–20min` là false positive hay khẳng định "
        "`20min` là optimum. Final dwell threshold vẫn phải dựa vào 02b."
    )

    return comp, diag, finding, supports, cannot_decide


def analyze_speed(path):
    comp = speed_guard_comparison(path)
    events = hard_speed_event_diagnostics(
        path, threshold_kmh=BASELINE["hard_speed_guard_kmh"]
    )

    baseline_boundaries = int(
        comp.loc[
            comp["speed_guard_kmh"] == BASELINE["hard_speed_guard_kmh"],
            "hard_speed_boundaries",
        ].iloc[0]
    )
    strict_boundaries = int(
        comp.loc[
            comp["speed_guard_kmh"] == 300.0,
            "hard_speed_boundaries",
        ].iloc[0]
    )

    if len(events):
        speeds = events["implied_speed_kmh"].round(1).tolist()
        finding = (
            f"`1200 km/h` bắt **{baseline_boundaries}** hard-speed boundary "
            f"với implied speed {speeds} km/h; `300 km/h` tạo "
            f"**{strict_boundaries}** boundary trên cùng trajectory."
        )
        supports = (
            "`1200 km/h` đang hoạt động như một **extreme pathology guard** trong case này, "
            "trong khi `300 km/h` can thiệp nhiều hơn."
        )
    else:
        finding = (
            "Selected trajectory không có event vượt `1200 km/h`; "
            f"`300 km/h` tạo **{strict_boundaries}** boundary."
        )
        supports = (
            "Case này chỉ cho thấy strict guard can thiệp nhiều hơn; "
            "không có event 1200 km/h để deep-dive."
        )

    cannot_decide = (
        "Case này không đủ để chứng minh `1200 km/h` là threshold tối ưu toàn dataset "
        "và không được diễn giải guard như transport-mode classifier."
    )

    return comp, events, finding, supports, cannot_decide


## 4.2 Population context — case study không đứng một mình

Case map giúp hiểu **behavior cụ thể**, còn deterministic sample cho biết thay đổi threshold có phổ biến hay chỉ xảy ra ở một trajectory.

Các bảng dưới đây **không tự chọn threshold**. Chúng chỉ trả lời:

- đổi parameter làm bao nhiêu trajectories thay đổi số stay;
- tổng số stay tăng/giảm ra sao;
- hard-speed event nằm ở tail như thế nào.

Population context ở 02a vẫn chỉ là diagnostic context. **Final CP1 freeze thuộc 02b**, nơi evaluation được thiết kế ở cấp population/user thay vì một selected trajectory.


In [ ]:

print("Gap sensitivity across deterministic sample")
display(
    population_sensitivity_table(
        case_scan, "gap", (120, 300, 600), 300
    )
)

print("Radius sensitivity across deterministic sample")
display(
    population_sensitivity_table(
        case_scan, "radius", (100, 200, 300), 200
    )
)

print("Dwell sensitivity across deterministic sample")
display(
    population_sensitivity_table(
        case_scan, "dwell", (600, 1200, 1800), 1200
    )
)

speed_population = pd.DataFrame([
    {
        "metric": "raw edges > 300 km/h",
        "count": int(case_scan["raw_edges_gt_300_kmh"].sum()),
        "trajectories_affected": int((case_scan["raw_edges_gt_300_kmh"] > 0).sum()),
    },
    {
        "metric": "raw edges > 1200 km/h",
        "count": int(case_scan["raw_edges_gt_1200_kmh"].sum()),
        "trajectories_affected": int((case_scan["raw_edges_gt_1200_kmh"] > 0).sum()),
    },
    {
        "metric": "baseline hard-speed boundaries after cleaning",
        "count": int(case_scan["hard_speed_boundaries"].sum()),
        "trajectories_affected": int((case_scan["hard_speed_boundaries"] > 0).sum()),
    },
])
print("Speed-tail context across deterministic sample")
display(speed_population)


## 5. Case A — `max_gap_s`: khi nào nên ngắt một trajectory?

`max_gap_s` là khoảng thời gian tối đa giữa hai GPS observations liên tiếp mà ta vẫn cho phép chúng thuộc cùng một continuous sequence.

Ta chỉ thay:

```text
max_gap_s = 120 / 300 / 600 s
```

Các giá trị **fix** trong Case A:

```text
same_second_radius_m   = 10 m
distance_threshold_m   = 200 m
min_dwell_s            = 1200 s = 20 min
hard_speed_guard_kmh   = 1200 km/h
```

Ba mức gap:

```text
120 s  → strict: dễ cắt trajectory khi sampling hơi thưa
300 s  → baseline hiện tại
600 s  → permissive: giữ continuity lâu hơn
```

Case này không dừng ở việc nhìn path có “mượt” hơn hay không. Ta kiểm tra thêm các **bridge chỉ xuất hiện khi tăng từ 300 lên 600 giây**:

```text
300 < Δt <= 600
```

Với mỗi bridge, notebook tính `distance` và `implied speed`.

> Nếu bridge hợp lý, ta chỉ kết luận rằng `600s` **hợp lý hơn cho continuity trên trajectory này**. Điều đó chưa đủ để tự động thay production baseline `300s`; quyết định đó phải qua population-level evidence ở 02b.


In [ ]:

gap_rows = selected_cases.loc[selected_cases["case"] == "gap"]

if gap_rows["file"].notna().any():
    gap_path = gap_rows["file"].iloc[0]
    print("Gap case:", gap_path)

    (
        gap_comp,
        gap_bridges,
        gap_finding,
        gap_supports,
        gap_cannot_decide,
    ) = analyze_gap(gap_path)

    display(gap_comp)
    plot_time_gaps(gap_path)

    print("Bridges introduced by moving 300s -> 600s:")
    if len(gap_bridges):
        display(gap_bridges)
    else:
        print("No consecutive raw edge with 300 < Δt <= 600 seconds.")

    if MAPTILER_API_KEY:
        gap_map = build_case_map(
            gap_path,
            configs=[
                {"name": "120 s — strict", "gap_s": 120, "radius_m": 200, "dwell_s": 1200, "show": False},
                {"name": "300 s — baseline", "gap_s": 300, "radius_m": 200, "dwell_s": 1200, "show": True},
                {"name": "600 s — permissive", "gap_s": 600, "radius_m": 200, "dwell_s": 1200, "show": False},
            ],
            title="Case A — continuity gap sensitivity",
            boundary_reason="temporal_gap",
        )
        gap_html = export_case_map(
            gap_map,
            "case_a_gap_sensitivity_deep_dive.html",
        )
    else:
        print("MAPTILER_API_KEY not set -> skip HTML map export; diagnostics still run.")

    _emit_case_finding(
        "A — max_gap_s",
        finding=gap_finding,
        supports=gap_supports,
        cannot_decide=gap_cannot_decide,
    )
else:
    print("No readable gap-sensitive case found under the deterministic rules.")


## 6. Case B — `distance_threshold_m`: một stay được phép “rộng” đến đâu?

`distance_threshold_m` là spatial tolerance dùng khi gom các observations thành một stay candidate.

Ta chỉ thay:

```text
distance_threshold_m = 100 / 200 / 300 m
```

Các giá trị **fix** trong Case B:

```text
same_second_radius_m   = 10 m
max_gap_s              = 300 s
min_dwell_s            = 1200 s = 20 min
hard_speed_guard_kmh   = 1200 km/h
```

Ba mức radius:

```text
100 m  → strict
200 m  → baseline hiện tại
300 m  → permissive
```

Ngoài số stay, notebook đo cho từng emitted stay:

- khoảng cách lớn nhất từ arrival anchor;
- khoảng cách lớn nhất quanh median center;
- start–end displacement;
- tổng path length bên trong stay;
- mức sử dụng radius (`max_anchor_distance / radius`);
- nếu `300m` tạo nhiều stay, khoảng cách và time gap giữa các stay liên tiếp.

Điểm quan trọng:

> Nếu một stay ở `300m` vẫn thỏa `dwell >= 20 min`, **không được tự gọi nó là false positive** chỉ vì `200m` không detect nó.

Case B chỉ được phép nói threshold nào strict hơn / permissive hơn và output thay đổi ra sao. Muốn chọn `200m` hay `300m` cần population-level evidence hoặc ground truth phù hợp.

> Stay radius là **tolerance của detector**, không phải kích thước vật lý thật của Home/Office.


In [ ]:

radius_rows = selected_cases.loc[selected_cases["case"] == "radius"]

if radius_rows["file"].notna().any():
    radius_path = radius_rows["file"].iloc[0]
    print("Radius case:", radius_path)

    (
        radius_comp,
        radius_diag,
        radius_pair_diag,
        radius_finding,
        radius_supports,
        radius_cannot_decide,
    ) = analyze_radius(
        radius_path,
        gap_s=int(BASELINE["max_gap_s"]),
    )

    display(radius_comp)

    print("Spatial diagnostics for emitted stays:")
    if len(radius_diag):
        display(
            radius_diag[
                [
                    "radius_m", "stay_id", "duration_min", "n_points",
                    "max_anchor_distance_m", "max_center_distance_m",
                    "start_end_distance_m", "path_length_m",
                    "radius_utilization",
                ]
            ].round(2)
        )
    else:
        print("No stay emitted for these radius settings.")

    print("Adjacent stay diagnostics at 300m:")
    if len(radius_pair_diag):
        display(radius_pair_diag.round(2))
    else:
        print("300m emitted fewer than two stays; no adjacent-pair comparison.")

    if MAPTILER_API_KEY:
        radius_map = build_case_map(
            radius_path,
            configs=[
                {"name": "100 m — strict", "gap_s": 300, "radius_m": 100, "dwell_s": 1200, "show": False},
                {"name": "200 m — baseline", "gap_s": 300, "radius_m": 200, "dwell_s": 1200, "show": True},
                {"name": "300 m — permissive", "gap_s": 300, "radius_m": 300, "dwell_s": 1200, "show": False},
            ],
            title="Case B — stay-radius sensitivity",
            draw_radius=True,
        )
        radius_html = export_case_map(
            radius_map,
            "case_b_radius_sensitivity_deep_dive.html",
        )
    else:
        print("MAPTILER_API_KEY not set -> skip HTML map export; diagnostics still run.")

    _emit_case_finding(
        "B — distance_threshold_m",
        finding=radius_finding,
        supports=radius_supports,
        cannot_decide=radius_cannot_decide,
    )
else:
    print("No readable radius-sensitive case found under the deterministic rules.")


### Cách đọc Case B

Ví dụ nếu output là:

```text
100m → 0 stay
200m → 1 stay
300m → 2 stays
```

thì kết luận đúng là:

- `100m` restrictive hơn trên trajectory này;
- `200m` và `300m` tạo behavior khác nhau;
- stay bổ sung ở `300m` nếu vẫn đủ dwell 20 phút thì **hợp lệ theo rule hiện tại**;
- **chỉ từ case này chưa đủ evidence để nói 200m tốt hơn 300m hoặc ngược lại**.

Notebook vì vậy không còn hard-code `200m` làm recommendation từ pattern `0 → 1 → 2`.


## 7. Case C — `min_dwell_s`: short pause hay sustained presence?

`min_dwell_s` quy định một candidate phải kéo dài tối thiểu bao lâu mới được emit thành stay.

Ta chỉ thay:

```text
min_dwell_s = 600 / 1200 / 1800 s
            = 10 / 20 / 30 min
```

Các giá trị **fix** trong Case C:

```text
same_second_radius_m   = 10 m
max_gap_s              = 300 s
distance_threshold_m   = 200 m
hard_speed_guard_kmh   = 1200 km/h
```

Ba mức dwell:

```text
10 min  → permissive: giữ nhiều short/medium pauses hơn
20 min  → baseline hiện tại
30 min  → conservative: chỉ giữ episode dài hơn
```

Case này tập trung vào **duration của các episode xuất hiện / biến mất**.

> Không có ground truth thì không gọi stay 10–20 phút là false positive. Ta chỉ có thể nói threshold nào phù hợp hơn với policy “sustained presence” đã định nghĩa.


In [ ]:

dwell_rows = selected_cases.loc[selected_cases["case"] == "dwell"]

if dwell_rows["file"].notna().any():
    dwell_path = dwell_rows["file"].iloc[0]
    print("Dwell case:", dwell_path)

    (
        dwell_comp,
        dwell_diag,
        dwell_finding,
        dwell_supports,
        dwell_cannot_decide,
    ) = analyze_dwell(
        dwell_path,
        gap_s=int(BASELINE["max_gap_s"]),
        radius_m=int(BASELINE["distance_threshold_m"]),
    )

    display(dwell_comp)

    print("Emitted episode durations:")
    if len(dwell_diag):
        display(dwell_diag)
    else:
        print("No stay emitted.")

    if MAPTILER_API_KEY:
        dwell_map = build_case_map(
            dwell_path,
            configs=[
                {"name": "10 min — permissive", "gap_s": 300, "radius_m": 200, "dwell_s": 600, "show": False},
                {"name": "20 min — baseline", "gap_s": 300, "radius_m": 200, "dwell_s": 1200, "show": True},
                {"name": "30 min — conservative", "gap_s": 300, "radius_m": 200, "dwell_s": 1800, "show": False},
            ],
            title="Case C — dwell-threshold sensitivity",
        )
        dwell_html = export_case_map(
            dwell_map,
            "case_c_dwell_sensitivity_deep_dive.html",
        )
    else:
        print("MAPTILER_API_KEY not set -> skip HTML map export; diagnostics still run.")

    _emit_case_finding(
        "C — min_dwell_s",
        finding=dwell_finding,
        supports=dwell_supports,
        cannot_decide=dwell_cannot_decide,
    )
else:
    print("No readable dwell-sensitive case found under the deterministic rules.")


## 8. Case D — `hard_speed_guard_kmh`: chặn jump vật lý bất thường

`hard_speed_guard_kmh` có semantics khác ba parameter trên.

Nó **không** dùng để phân loại walking / car / train / plane. Nó chỉ tạo một continuity boundary khi hai observations liên tiếp ngụ ý tốc độ cực đoan đến mức path không nên được nối thẳng qua.

Ta chỉ thay:

```text
hard_speed_guard_kmh = 300 / 1200 / off
```

Các giá trị **fix** trong Case D:

```text
same_second_radius_m   = 10 m
max_gap_s              = 300 s
distance_threshold_m   = 200 m
min_dwell_s            = 1200 s = 20 min
```

Ba mức:

```text
300 km/h   → strict guard
1200 km/h  → baseline pathology guard
guard off  → không dùng speed để ngắt continuity
```

Deep dive tính chính xác:

```text
Δt
distance
implied speed
```

cho edge bị `1200 km/h` bắt, rồi so số boundary nếu guard chặt hơn hoặc tắt hoàn toàn.

> Case này có thể kiểm tra **semantics của pathology guard**, nhưng không tự chứng minh 1200 km/h là threshold tối ưu toàn dataset.


In [ ]:

hard_rows = selected_cases.loc[selected_cases["case"] == "hard_speed"]

if hard_rows["file"].notna().any():
    hard_path = hard_rows["file"].iloc[0]
    print("Hard-speed case:", hard_path)

    (
        speed_comp,
        speed_events,
        speed_finding,
        speed_supports,
        speed_cannot_decide,
    ) = analyze_speed(hard_path)

    display(speed_comp)

    print("Baseline 1200 km/h event diagnostics:")
    if len(speed_events):
        display(speed_events.round(2))
    else:
        print("No 1200 km/h hard-speed event on this case.")

    if MAPTILER_API_KEY:
        hard_map = build_case_map(
            hard_path,
            configs=[
                {
                    "name": "300 km/h — strict guard",
                    "gap_s": 300, "radius_m": 200, "dwell_s": 1200,
                    "speed_guard_kmh": 300.0, "show": False,
                },
                {
                    "name": "1200 km/h — baseline guard",
                    "gap_s": 300, "radius_m": 200, "dwell_s": 1200,
                    "speed_guard_kmh": 1200.0, "show": True,
                },
                {
                    "name": "guard off — permissive",
                    "gap_s": 300, "radius_m": 200, "dwell_s": 1200,
                    "speed_guard_kmh": 1e9, "show": False,
                },
            ],
            title="Case D — hard-speed guard sensitivity",
            boundary_reason="hard_speed_guard",
        )
        hard_html = export_case_map(
            hard_map,
            "case_d_hard_speed_guard_deep_dive.html",
        )
    else:
        print("MAPTILER_API_KEY not set -> skip HTML map export; diagnostics still run.")

    _emit_case_finding(
        "D — hard_speed_guard_kmh",
        finding=speed_finding,
        supports=speed_supports,
        cannot_decide=speed_cannot_decide,
    )
else:
    print(
        "No hard-speed event found in the deterministic case sample. "
        "Do not lower the threshold only to manufacture a boundary."
    )


## 8.1 `same_second_radius_m = 10m` không nằm trong bốn case này

`same_second_radius_m` có semantics riêng:

```text
safe-to-collapse same-second group
≠
trajectory continuity / stay tolerance / dwell / hard-speed policy
```

Concern này tiếp tục được audit riêng trong `02c_same_second_transport_audit.ipynb`.

Notebook 02a **không freeze lại** `same_second_radius_m`, và cũng không dùng bốn case A/B/C/D để thay đổi semantics đã audit ở 02c.


## 9. Cross-case summary — không auto-freeze config

Bốn case trên là sensitivity theo từng trục và có thể tương tác với nhau. Ví dụ đổi `max_gap_s` có thể làm sequence structure khác đi, từ đó làm radius/dwell produce stay khác.

Tuy nhiên 02a **không sequentially chọn parameter rồi ghép thành `FINAL_CONFIG`** nữa, vì cách đó có thể khuếch đại một kết luận local từ một trajectory thành production decision.

Thay vào đó phần cuối chỉ làm ba việc:

```text
1. tổng hợp finding của Case A/B/C/D;
2. giữ nguyên BASELINE dùng để isolate các case;
3. ghi rõ handoff:
   02a → explain behavior
   02b → freeze CP1 ở population level
   02c → audit same-second semantics
   03  → consume CP1 stays đã freeze
```

Nếu 02b sau này thực sự đổi CP1 baseline, **03 phải regenerate stay cache và rerun downstream Home/Office analysis**.


In [ ]:

# 02a intentionally does NOT build a proposed/final production config.
# It summarizes selected-case evidence and hands the freeze decision to 02b.

summary_rows = []
for case_name, info in CASE_FINDINGS.items():
    summary_rows.append({
        "case": case_name,
        "finding": info["finding"],
        "supports": info["supports"],
        "cannot_decide": info["cannot_decide"],
    })

case_summary = pd.DataFrame(summary_rows)

display(Markdown("### Case-study evidence summary"))
if len(case_summary):
    display(case_summary)
else:
    print("No case findings were generated.")

baseline_table = pd.DataFrame([
    {
        "parameter": "same_second_radius_m",
        "baseline_value": BASELINE["same_second_radius_m"],
        "unit": "m",
        "02a_role": "fixed; audited separately in 02c",
    },
    {
        "parameter": "max_gap_s",
        "baseline_value": BASELINE["max_gap_s"],
        "unit": "s",
        "02a_role": "sensitivity case only; no auto-freeze",
    },
    {
        "parameter": "distance_threshold_m",
        "baseline_value": BASELINE["distance_threshold_m"],
        "unit": "m",
        "02a_role": "sensitivity case only; no auto-freeze",
    },
    {
        "parameter": "min_dwell_s",
        "baseline_value": BASELINE["min_dwell_s"],
        "unit": "s",
        "02a_role": "sensitivity case only; no auto-freeze",
    },
    {
        "parameter": "hard_speed_guard_kmh",
        "baseline_value": BASELINE["hard_speed_guard_kmh"],
        "unit": "km/h",
        "02a_role": "pathology-guard case only; no auto-freeze",
    },
])

display(Markdown("### Baseline remains unchanged inside 02a"))
display(baseline_table)

display(
    Markdown(
        "### Handoff\n\n"
        "**02a không tạo `FINAL_CONFIG`.**\n\n"
        "- dùng Case A/B/C/D để hiểu mechanics và phát hiện assumption sai;\n"
        "- dùng **02b** để freeze CP1 bằng population-level sensitivity;\n"
        "- giữ **02c** cho same-second semantic audit;\n"
        "- **03** chỉ được consume stay inventory sinh từ CP1 config đã freeze.\n\n"
        "Nếu 02b thay đổi bất kỳ CP1 parameter nào, cache/count downstream của 03 "
        "phải được regenerate trước khi tiếp tục Home/Office analysis."
    )
)



## 10. Giới hạn và handoff

Notebook 02a cho phép hiểu **parameter behavior có evidence ở cấp trajectory**, nhưng không chứng minh accuracy theo ground truth.

Cụ thể:

- `600s` có thể hợp lý hơn `300s` cho continuity của Case A nếu các bridge mới vẫn spatially plausible; điều đó **không tự động đổi production gap**;
- `100/200/300m` chỉ cho thấy detector strict/permissive ra sao; nếu cả `200m` và `300m` tạo stay đủ dwell thì Case B **không được tự chọn winner**;
- `10/20/30min` biểu diễn ba dwell policies; “20 phút” chỉ là middle sustained-presence policy nếu không có ground truth;
- `1200 km/h` là pathology guard, không phải transport-mode threshold;
- số stay nhiều hơn hoặc ít hơn tự nó không phải accuracy metric;
- `distance_threshold_m = 200m` ở CP1 **khác semantics** với `location_max_diameter_m = 200m` ở CP2/03.

### Handoff chuẩn

```text
02a: selected case studies / mechanics / diagnostics
  ↓
02b: population-level sensitivity → freeze CP1
  ↓
02c: independent same-second semantic audit
  ↓
03: materialize stays from frozen CP1 → Home/Office
```

Nếu CP1 config thay đổi sau 02b, mọi cache/count downstream trong 03 phải được xem là stale và regenerate trước khi dùng tiếp.
